In [2]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

In [5]:
from google.colab import files

uploaded = files.upload()

Saving features.csv to features.csv
Saving stores.csv to stores.csv
Saving train.csv to train.csv


In [6]:
train = pd.read_csv("train.csv")
features = pd.read_csv("features.csv")
stores = pd.read_csv("stores.csv")

print(train.shape)
print(features.shape)
print(stores.shape)

(421570, 5)
(8190, 12)
(45, 3)


In [8]:
train = pd.read_csv("train.csv")
features = pd.read_csv("features.csv")
stores = pd.read_csv("stores.csv")

print(train.shape)
print(features.shape)
print(stores.shape)
print(train.head())
print(features.head())
print(stores.head())

(421570, 5)
(8190, 12)
(45, 3)
   Store  Dept        Date  Weekly_Sales  IsHoliday
0      1     1  2010-02-05      24924.50      False
1      1     1  2010-02-12      46039.49       True
2      1     1  2010-02-19      41595.55      False
3      1     1  2010-02-26      19403.54      False
4      1     1  2010-03-05      21827.90      False
   Store        Date  Temperature  ...         CPI  Unemployment  IsHoliday
0      1  2010-02-05        42.31  ...  211.096358         8.106      False
1      1  2010-02-12        38.51  ...  211.242170         8.106       True
2      1  2010-02-19        39.93  ...  211.289143         8.106      False
3      1  2010-02-26        46.63  ...  211.319643         8.106      False
4      1  2010-03-05        46.50  ...  211.350143         8.106      False

[5 rows x 12 columns]
   Store Type    Size
0      1    A  151315
1      2    A  202307
2      3    B   37392
3      4    A  205863
4      5    B   34875


In [9]:
train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

print(train.dtypes)
print(features.dtypes)

Store                    int64
Dept                     int64
Date            datetime64[ns]
Weekly_Sales           float64
IsHoliday                 bool
dtype: object
Store                    int64
Date            datetime64[ns]
Temperature            float64
Fuel_Price             float64
MarkDown1              float64
MarkDown2              float64
MarkDown3              float64
MarkDown4              float64
MarkDown5              float64
CPI                    float64
Unemployment           float64
IsHoliday                 bool
dtype: object


In [10]:
df = train.merge(
    features,
    on=["Store", "Date", "IsHoliday"],
    how="left"
)

df = df.merge(
    stores,
    on="Store",
    how="left"
)

print(df.shape)
df.head()

(421570, 16)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [11]:
# Replace missing markdown values with 0
markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

df[markdown_cols] = df[markdown_cols].fillna(0)

# Remove negative sales because we are forecasting demand
df = df[df["Weekly_Sales"] >= 0]

# Create SKU_ID: one Store + Dept combination = one planning unit
df["SKU_ID"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)

print(df.shape)
df.head()

(420285, 17)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,SKU_ID
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315,1_1
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315,1_1
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315,1_1
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315,1_1
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315,1_1


In [12]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Date"].dt.quarter

# Convert holiday True/False into 1/0
df["IsHoliday"] = df["IsHoliday"].astype(int)

df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,SKU_ID,Year,Month,Week,Quarter
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315,1_1,2010,2,5,1
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315,1_1,2010,2,6,1
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315,1_1,2010,2,7,1
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315,1_1,2010,2,8,1
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315,1_1,2010,3,9,1


In [13]:
df = pd.get_dummies(df, columns=["Type"], drop_first=True, dtype=int)

df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Size,SKU_ID,Year,Month,Week,Quarter,Type_B,Type_C
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,151315,1_1,2010,2,5,1,0,0
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,151315,1_1,2010,2,6,1,0,0
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,151315,1_1,2010,2,7,1,0,0
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,151315,1_1,2010,2,8,1,0,0
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,151315,1_1,2010,3,9,1,0,0


In [14]:
df = df.sort_values(["Store", "Dept", "Date"])

df[["Store", "Dept", "Date", "Weekly_Sales"]].head(15)

,Store,Dept,Date,Weekly_Sales
0,1,1,2010-02-05,24924.50
1,1,1,2010-02-12,46039.49
2,1,1,2010-02-19,41595.55
3,1,1,2010-02-26,19403.54
4,1,1,2010-03-05,21827.90
5,1,1,2010-03-12,21043.39
6,1,1,2010-03-19,22136.64
7,1,1,2010-03-26,26229.21
8,1,1,2010-04-02,57258.43
9,1,1,2010-04-09,42960.91


In [15]:
df["Sales_Lag_1"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(1)
df["Sales_Lag_2"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(2)
df["Sales_Lag_4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(4)
df["Sales_Lag_8"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(8)
df["Sales_Lag_12"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(12)

df[["Store", "Dept", "Date", "Weekly_Sales", "Sales_Lag_1", "Sales_Lag_4", "Sales_Lag_12"]].head(20)

,Store,Dept,Date,Weekly_Sales,Sales_Lag_1,Sales_Lag_4,Sales_Lag_12
0,1,1,2010-02-05,24924.50,NaN,NaN,NaN
1,1,1,2010-02-12,46039.49,24924.50,NaN,NaN
2,1,1,2010-02-19,41595.55,46039.49,NaN,NaN
3,1,1,2010-02-26,19403.54,41595.55,NaN,NaN
4,1,1,2010-03-05,21827.90,19403.54,24924.50,NaN
5,1,1,2010-03-12,21043.39,21827.90,46039.49,NaN
6,1,1,2010-03-19,22136.64,21043.39,41595.55,NaN
7,1,1,2010-03-26,26229.21,22136.64,19403.54,NaN
8,1,1,2010-04-02,57258.43,26229.21,21827.90,NaN
9,1,1,2010-04-09,42960.91,57258.43,21043.39,NaN


In [16]:
df["Rolling_Mean_4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
    lambda x: x.shift(1).rolling(window=4).mean()
)

df["Rolling_Mean_8"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
    lambda x: x.shift(1).rolling(window=8).mean()
)

df["Rolling_Std_4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
    lambda x: x.shift(1).rolling(window=4).std()
)

df["Rolling_Std_8"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
    lambda x: x.shift(1).rolling(window=8).std()
)

df[["Store", "Dept", "Date", "Weekly_Sales", "Rolling_Mean_4", "Rolling_Std_4"]].head(20)

,Store,Dept,Date,Weekly_Sales,Rolling_Mean_4,Rolling_Std_4
0,1,1,2010-02-05,24924.50,NaN,NaN
1,1,1,2010-02-12,46039.49,NaN,NaN
2,1,1,2010-02-19,41595.55,NaN,NaN
3,1,1,2010-02-26,19403.54,NaN,NaN
4,1,1,2010-03-05,21827.90,32990.7700,12832.106391
5,1,1,2010-03-12,21043.39,32216.6200,13554.047185
6,1,1,2010-03-19,22136.64,25967.5950,10467.484020
7,1,1,2010-03-26,26229.21,21102.8675,1222.784968
8,1,1,2010-04-02,57258.43,22809.2850,2325.929203
9,1,1,2010-04-09,42960.91,31666.9175,17206.391261


In [17]:
df = df.dropna()

print(df.shape)
df.head()

(381870, 31)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Size,SKU_ID,Year,Month,Week,Quarter,Type_B,Type_C,Sales_Lag_1,Sales_Lag_2,Sales_Lag_4,Sales_Lag_8,Sales_Lag_12,Rolling_Mean_4,Rolling_Mean_8,Rolling_Std_4,Rolling_Std_8
12,1,1,2010-04-30,16555.11,0,67.41,2.780,0.0,0.0,0.0,0.0,0.0,210.389546,7.808,151315,1_1,2010,4,17,2,0,0,16145.35,17596.96,57258.43,21827.90,24924.50,33490.4125,28149.84875,20067.070906,14404.685813
13,1,1,2010-05-07,17413.94,0,72.55,2.835,0.0,0.0,0.0,0.0,0.0,210.339968,7.808,151315,1_1,2010,5,18,2,0,0,16555.11,16145.35,42960.91,21043.39,46039.49,23314.5825,27490.75000,13111.798178,14849.052182
14,1,1,2010-05-14,18926.74,0,74.78,2.854,0.0,0.0,0.0,0.0,0.0,210.337426,7.808,151315,1_1,2010,5,19,2,0,0,17413.94,16555.11,17596.96,22136.64,41595.55,16927.8400,27037.06875,691.672619,15127.021661
15,1,1,2010-05-21,14773.04,0,76.44,2.826,0.0,0.0,0.0,0.0,0.0,210.617093,7.808,151315,1_1,2010,5,20,2,0,0,18926.74,17413.94,16145.35,26229.21,19403.54,17260.2850,26635.83125,1230.316214,15316.950408
16,1,1,2010-05-28,15580.43,0,80.44,2.759,0.0,0.0,0.0,0.0,0.0,210.896761,7.808,151315,1_1,2010,5,21,2,0,0,14773.04,18926.74,16555.11,57258.43,21827.90,16917.2075,25203.81000,1733.352524,15885.383152


In [18]:
target = "Weekly_Sales"

features_list = [
    "Store", "Dept", "Size",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "IsHoliday",
    "Year", "Month", "Week", "Quarter",
    "Sales_Lag_1", "Sales_Lag_2", "Sales_Lag_4", "Sales_Lag_8", "Sales_Lag_12",
    "Rolling_Mean_4", "Rolling_Mean_8", "Rolling_Std_4", "Rolling_Std_8"
]

# Add store type columns if they exist
for col in ["Type_B", "Type_C"]:
    if col in df.columns:
        features_list.append(col)

X = df[features_list]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures used:")
print(features_list)

X shape: (381870, 28)
y shape: (381870,)

Features used:
['Store', 'Dept', 'Size', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'IsHoliday', 'Year', 'Month', 'Week', 'Quarter', 'Sales_Lag_1', 'Sales_Lag_2', 'Sales_Lag_4', 'Sales_Lag_8', 'Sales_Lag_12', 'Rolling_Mean_4', 'Rolling_Mean_8', 'Rolling_Std_4', 'Rolling_Std_8', 'Type_B', 'Type_C']


In [19]:
cutoff_date = df["Date"].max() - pd.Timedelta(weeks=8)

train_df = df[df["Date"] <= cutoff_date]
test_df = df[df["Date"] > cutoff_date]

X_train = train_df[features_list]
y_train = train_df[target]

X_test = test_df[features_list]
y_test = test_df[target]

print("Max date in full data:", df["Date"].max())
print("Cutoff date:", cutoff_date)

print("\nTraining date range:", train_df["Date"].min(), "to", train_df["Date"].max())
print("Testing date range:", test_df["Date"].min(), "to", test_df["Date"].max())

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Max date in full data: 2012-10-26 00:00:00
Cutoff date: 2012-08-31 00:00:00

Training date range: 2010-04-30 00:00:00 to 2012-08-31 00:00:00
Testing date range: 2012-09-07 00:00:00 to 2012-10-26 00:00:00

X_train shape: (358266, 28)
X_test shape: (23604, 28)
y_train shape: (358266,)
y_test shape: (23604,)


In [20]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    tree_method="hist"
)

model.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Make a copy of test data
test_df = test_df.copy()

# Predict sales
test_df["Predicted_Sales"] = model.predict(X_test)

# Sales cannot be negative, so convert negative predictions to 0
test_df["Predicted_Sales"] = test_df["Predicted_Sales"].clip(lower=0)

# Calculate errors
test_df["Forecast_Error"] = test_df["Weekly_Sales"] - test_df["Predicted_Sales"]
test_df["Abs_Error"] = abs(test_df["Forecast_Error"])

# Evaluation metrics
mae = mean_absolute_error(test_df["Weekly_Sales"], test_df["Predicted_Sales"])
rmse = np.sqrt(mean_squared_error(test_df["Weekly_Sales"], test_df["Predicted_Sales"]))

# MAPE: avoid division by zero
non_zero_actuals = test_df["Weekly_Sales"] != 0

mape = np.mean(
    np.abs(
        (test_df.loc[non_zero_actuals, "Weekly_Sales"] - test_df.loc[non_zero_actuals, "Predicted_Sales"])
        / test_df.loc[non_zero_actuals, "Weekly_Sales"]
    )
) * 100

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("MAPE:", round(mape, 2), "%")

test_df[["Store", "Dept", "Date", "Weekly_Sales", "Predicted_Sales", "Forecast_Error", "Abs_Error"]].head()

MAE: 1316.18
RMSE: 2841.65
MAPE: 507.84 %


,Store,Dept,Date,Weekly_Sales,Predicted_Sales,Forecast_Error,Abs_Error
135,1,1,2012-09-07,18322.37,17359.859375,962.510625,962.510625
136,1,1,2012-09-14,19616.22,18466.982422,1149.237578,1149.237578
137,1,1,2012-09-21,19251.50,20070.406250,-818.906250,818.906250
138,1,1,2012-09-28,18947.81,19255.746094,-307.936094,307.936094
139,1,1,2012-10-05,21904.47,21104.568359,799.901641,799.901641


In [22]:
# Better metric for retail forecasting: WMAPE
wmape = (
    test_df["Abs_Error"].sum() / test_df["Weekly_Sales"].sum()
) * 100

# Also calculate filtered MAPE only where actual sales are meaningful
filtered_df = test_df[test_df["Weekly_Sales"] > 1000]

filtered_mape = np.mean(
    np.abs(
        (filtered_df["Weekly_Sales"] - filtered_df["Predicted_Sales"])
        / filtered_df["Weekly_Sales"]
    )
) * 100

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("Raw MAPE:", round(mape, 2), "%")
print("Filtered MAPE Weekly_Sales > 1000:", round(filtered_mape, 2), "%")
print("WMAPE:", round(wmape, 2), "%")

MAE: 1316.18
RMSE: 2841.65
Raw MAPE: 507.84 %
Filtered MAPE Weekly_Sales > 1000: 12.79 %
WMAPE: 8.51 %


In [23]:
import os

# Create outputs folder
os.makedirs("outputs", exist_ok=True)

# Forecast results for Power BI
forecast_results = test_df[
    [
        "Store", "Dept", "SKU_ID", "Date",
        "Weekly_Sales", "Predicted_Sales",
        "Forecast_Error", "Abs_Error",
        "IsHoliday", "Size"
    ]
].copy()

forecast_results.to_csv("outputs/forecast_results.csv", index=False)

# Model metrics for Power BI / README
metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "Raw MAPE", "Filtered MAPE", "WMAPE"],
    "Value": [mae, rmse, mape, filtered_mape, wmape]
})

metrics.to_csv("outputs/model_metrics.csv", index=False)

print("Files exported successfully!")
print(forecast_results.shape)
print(metrics)

Files exported successfully!
(23604, 10)
          Metric        Value
0            MAE  1316.176281
1           RMSE  2841.653336
2       Raw MAPE   507.843836
3  Filtered MAPE    12.790391
4          WMAPE     8.511311


In [24]:
sku_summary = df.groupby(["Store", "Dept", "SKU_ID"]).agg(
    Total_Sales=("Weekly_Sales", "sum"),
    Avg_Weekly_Demand=("Weekly_Sales", "mean"),
    Demand_StdDev=("Weekly_Sales", "std"),
    Weeks_Observed=("Date", "nunique"),
    Store_Size=("Size", "max")
).reset_index()

# If any SKU has missing std deviation, replace it with 0
sku_summary["Demand_StdDev"] = sku_summary["Demand_StdDev"].fillna(0)

print(sku_summary.shape)
sku_summary.head()

(3115, 8)


,Store,Dept,SKU_ID,Total_Sales,Avg_Weekly_Demand,Demand_StdDev,Weeks_Observed,Store_Size
0,1,1,1_1,2862243.31,21849.185573,9236.989912,131,151315
1,1,2,1_2,6043987.84,46137.311756,3520.544777,131,151315
2,1,3,1_3,1752032.41,13374.293206,9059.639078,131,151315
3,1,4,1_4,4857805.87,37082.487557,2970.767236,131,151315
4,1,5,1_5,3165461.09,24163.825115,11660.582091,131,151315


In [25]:
forecast_avg = forecast_results.groupby(["Store", "Dept", "SKU_ID"]).agg(
    Forecasted_Weekly_Demand=("Predicted_Sales", "mean")
).reset_index()

inventory = sku_summary.merge(
    forecast_avg,
    on=["Store", "Dept", "SKU_ID"],
    how="left"
)

# If forecast is missing for any SKU, use historical average demand
inventory["Forecasted_Weekly_Demand"] = inventory["Forecasted_Weekly_Demand"].fillna(
    inventory["Avg_Weekly_Demand"]
)

print(inventory.shape)
inventory.head()

(3115, 9)


,Store,Dept,SKU_ID,Total_Sales,Avg_Weekly_Demand,Demand_StdDev,Weeks_Observed,Store_Size,Forecasted_Weekly_Demand
0,1,1,1_1,2862243.31,21849.185573,9236.989912,131,151315,21426.912109
1,1,2,1_2,6043987.84,46137.311756,3520.544777,131,151315,45409.812500
2,1,3,1_3,1752032.41,13374.293206,9059.639078,131,151315,16773.699219
3,1,4,1_4,4857805.87,37082.487557,2970.767236,131,151315,36413.222656
4,1,5,1_5,3165461.09,24163.825115,11660.582091,131,151315,22089.019531


In [26]:
import numpy as np

np.random.seed(42)

inventory["Lead_Time_Weeks"] = np.random.choice(
    [1, 2, 3, 4],
    size=len(inventory),
    p=[0.25, 0.35, 0.25, 0.15]
)

print(inventory.shape)
inventory[["Store", "Dept", "SKU_ID", "Forecasted_Weekly_Demand", "Lead_Time_Weeks"]].head(10)

(3115, 10)


,Store,Dept,SKU_ID,Forecasted_Weekly_Demand,Lead_Time_Weeks
0,1,1,1_1,21426.912109,2
1,1,2,1_2,45409.812500,4
2,1,3,1_3,16773.699219,3
3,1,4,1_4,36413.222656,2
4,1,5,1_5,22089.019531,1
5,1,6,1_6,3755.226074,1
6,1,7,1_7,20208.314453,1
7,1,8,1_8,37040.996094,4
8,1,9,1_9,36717.449219,3
9,1,10,1_10,29144.607422,3


In [27]:
Z_SCORE = 1.65

inventory["Safety_Stock_Value"] = (
    Z_SCORE * inventory["Demand_StdDev"] * np.sqrt(inventory["Lead_Time_Weeks"])
)

print(inventory.shape)
inventory[
    [
        "Store", "Dept", "SKU_ID",
        "Demand_StdDev", "Lead_Time_Weeks",
        "Safety_Stock_Value"
    ]
].head(10)

(3115, 11)


,Store,Dept,SKU_ID,Demand_StdDev,Lead_Time_Weeks,Safety_Stock_Value
0,1,1,1_1,9236.989912,2,21554.076075
1,1,2,1_2,3520.544777,4,11617.797765
2,1,3,1_3,9059.639078,3,25891.396049
3,1,4,1_4,2970.767236,2,6932.143871
4,1,5,1_5,11660.582091,1,19239.960450
5,1,6,1_6,3418.267550,1,5640.141458
6,1,7,1_7,19590.920014,1,32325.018024
7,1,8,1_8,2492.272681,4,8224.499846
8,1,9,1_9,8505.643090,3,24308.139871
9,1,10,1_10,3625.215136,3,10360.443725


In [28]:
inventory["Reorder_Point_Value"] = (
    inventory["Forecasted_Weekly_Demand"] * inventory["Lead_Time_Weeks"]
    + inventory["Safety_Stock_Value"]
)

print(inventory.shape)

inventory[
    [
        "Store", "Dept", "SKU_ID",
        "Forecasted_Weekly_Demand",
        "Lead_Time_Weeks",
        "Safety_Stock_Value",
        "Reorder_Point_Value"
    ]
].head(10)

(3115, 12)


,Store,Dept,SKU_ID,Forecasted_Weekly_Demand,Lead_Time_Weeks,Safety_Stock_Value,Reorder_Point_Value
0,1,1,1_1,21426.912109,2,21554.076075,64407.900294
1,1,2,1_2,45409.812500,4,11617.797765,193257.047765
2,1,3,1_3,16773.699219,3,25891.396049,76212.493705
3,1,4,1_4,36413.222656,2,6932.143871,79758.589184
4,1,5,1_5,22089.019531,1,19239.960450,41328.979981
5,1,6,1_6,3755.226074,1,5640.141458,9395.367532
6,1,7,1_7,20208.314453,1,32325.018024,52533.332477
7,1,8,1_8,37040.996094,4,8224.499846,156388.484221
8,1,9,1_9,36717.449219,3,24308.139871,134460.487527
9,1,10,1_10,29144.607422,3,10360.443725,97794.265991


In [29]:
np.random.seed(42)

# Simulate how many weeks of stock each SKU currently has
inventory["Stock_Cover_Weeks"] = np.random.randint(1, 9, size=len(inventory))

# Simulated current stock value
inventory["Current_Stock_Value"] = (
    inventory["Avg_Weekly_Demand"] * inventory["Stock_Cover_Weeks"]
)

# Compare current stock with reorder point
inventory["Stock_Status"] = np.where(
    inventory["Current_Stock_Value"] < inventory["Reorder_Point_Value"],
    "Reorder Required",
    "Stock Sufficient"
)

print(inventory.shape)

inventory[
    [
        "Store", "Dept", "SKU_ID",
        "Avg_Weekly_Demand",
        "Stock_Cover_Weeks",
        "Current_Stock_Value",
        "Reorder_Point_Value",
        "Stock_Status"
    ]
].head(10)

(3115, 15)


,Store,Dept,SKU_ID,Avg_Weekly_Demand,Stock_Cover_Weeks,Current_Stock_Value,Reorder_Point_Value,Stock_Status
0,1,1,1_1,21849.185573,7,152944.299008,64407.900294,Stock Sufficient
1,1,2,1_2,46137.311756,4,184549.247023,193257.047765,Reorder Required
2,1,3,1_3,13374.293206,5,66871.466031,76212.493705,Reorder Required
3,1,4,1_4,37082.487557,7,259577.412901,79758.589184,Stock Sufficient
4,1,5,1_5,24163.825115,3,72491.475344,41328.979981,Stock Sufficient
5,1,6,1_6,4775.871923,8,38206.975385,9395.367532,Stock Sufficient
6,1,7,1_7,25003.361298,5,125016.806489,52533.332477,Stock Sufficient
7,1,8,1_8,35703.725344,5,178518.626718,156388.484221,Stock Sufficient
8,1,9,1_9,28491.540229,7,199440.781603,134460.487527,Stock Sufficient
9,1,10,1_10,31037.021145,2,62074.042290,97794.265991,Reorder Required


In [30]:
# Sort SKUs by total sales from highest to lowest
inventory = inventory.sort_values("Total_Sales", ascending=False)

# Calculate cumulative sales
inventory["Cumulative_Sales"] = inventory["Total_Sales"].cumsum()

# Calculate cumulative sales percentage
inventory["Cumulative_Sales_Percentage"] = (
    inventory["Cumulative_Sales"] / inventory["Total_Sales"].sum()
)

# Create ABC class
def abc_class(x):
    if x <= 0.80:
        return "A"
    elif x <= 0.95:
        return "B"
    else:
        return "C"

inventory["ABC_Class"] = inventory["Cumulative_Sales_Percentage"].apply(abc_class)

print(inventory.shape)

inventory[
    [
        "Store", "Dept", "SKU_ID",
        "Total_Sales",
        "Cumulative_Sales_Percentage",
        "ABC_Class",
        "Stock_Status"
    ]
].head(15)

(3115, 18)


,Store,Dept,SKU_ID,Total_Sales,Cumulative_Sales_Percentage,ABC_Class,Stock_Status
1001,14,92,14_92,23758069.98,0.003847,A,Stock Sufficient
1437,20,92,20_92,21665113.02,0.007354,A,Stock Sufficient
141,2,92,2_92,21626728.16,0.010856,A,Reorder Required
927,13,92,13_92,21358729.13,0.014314,A,Stock Sufficient
282,4,92,4_92,21014052.32,0.017716,A,Stock Sufficient
1440,20,95,20_95,19739332.85,0.020912,A,Reorder Required
285,4,95,4_95,19496767.91,0.024068,A,Stock Sufficient
1942,27,92,27_92,19103305.64,0.027161,A,Stock Sufficient
144,2,95,2_95,18848731.38,0.030213,A,Stock Sufficient
1004,14,95,14_95,18787642.70,0.033255,A,Reorder Required


In [31]:
def action_recommendation(row):
    if row["Stock_Status"] == "Reorder Required" and row["ABC_Class"] == "A":
        return "Urgent Reorder - High Priority SKU"
    elif row["Stock_Status"] == "Reorder Required" and row["ABC_Class"] == "B":
        return "Reorder Required - Medium Priority"
    elif row["Stock_Status"] == "Reorder Required" and row["ABC_Class"] == "C":
        return "Reorder Required - Low Priority"
    elif row["ABC_Class"] == "C" and row["Stock_Cover_Weeks"] >= 6:
        return "Review Overstock Risk"
    else:
        return "Stock Sufficient"

inventory["Action_Recommendation"] = inventory.apply(action_recommendation, axis=1)

print(inventory.shape)

inventory[
    [
        "Store", "Dept", "SKU_ID",
        "ABC_Class",
        "Stock_Status",
        "Stock_Cover_Weeks",
        "Action_Recommendation"
    ]
].head(20)

(3115, 19)


,Store,Dept,SKU_ID,ABC_Class,Stock_Status,Stock_Cover_Weeks,Action_Recommendation
1001,14,92,14_92,A,Stock Sufficient,7,Stock Sufficient
1437,20,92,20_92,A,Stock Sufficient,5,Stock Sufficient
141,2,92,2_92,A,Reorder Required,1,Urgent Reorder - High Priority SKU
927,13,92,13_92,A,Stock Sufficient,4,Stock Sufficient
282,4,92,4_92,A,Stock Sufficient,6,Stock Sufficient
1440,20,95,20_95,A,Reorder Required,1,Urgent Reorder - High Priority SKU
285,4,95,4_95,A,Stock Sufficient,3,Stock Sufficient
1942,27,92,27_92,A,Stock Sufficient,6,Stock Sufficient
144,2,95,2_95,A,Stock Sufficient,8,Stock Sufficient
1004,14,95,14_95,A,Reorder Required,1,Urgent Reorder - High Priority SKU


In [32]:
import os

os.makedirs("outputs", exist_ok=True)

inventory.to_csv("outputs/inventory_planning.csv", index=False)

print("Inventory planning file exported successfully!")
print(inventory.shape)

inventory.head()

Inventory planning file exported successfully!
(3115, 19)


,Store,Dept,SKU_ID,Total_Sales,Avg_Weekly_Demand,Demand_StdDev,Weeks_Observed,Store_Size,Forecasted_Weekly_Demand,Lead_Time_Weeks,Safety_Stock_Value,Reorder_Point_Value,Stock_Cover_Weeks,Current_Stock_Value,Stock_Status,Cumulative_Sales,Cumulative_Sales_Percentage,ABC_Class,Action_Recommendation
1001,14,92,14_92,23758069.98,181359.312824,23260.609979,131,200898,160118.875000,2,54277.525666,374515.275666,7,1.269515e+06,Stock Sufficient,2.375807e+07,0.003847,A,Stock Sufficient
1437,20,92,20_92,21665113.02,165382.542137,20049.359398,131,203742,169743.125000,3,57298.740077,566528.115077,5,8.269127e+05,Stock Sufficient,4.542318e+07,0.007354,A,Stock Sufficient
141,2,92,2_92,21626728.16,165089.527939,20310.399387,131,202307,165392.390625,2,47393.349747,378178.130997,1,1.650895e+05,Reorder Required,6.704991e+07,0.010856,A,Urgent Reorder - High Priority SKU
927,13,92,13_92,21358729.13,163043.733817,18692.483920,131,219622,168272.609375,2,43618.021052,380163.239802,4,6.521749e+05,Stock Sufficient,8.840864e+07,0.014314,A,Stock Sufficient
282,4,92,4_92,21014052.32,160412.613130,19577.991048,131,205863,168240.671875,2,45684.309766,382165.653516,6,9.624757e+05,Stock Sufficient,1.094227e+08,0.017716,A,Stock Sufficient


In [33]:
# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": features_list,
    "Importance": model.feature_importances_
})

# Sort from most important to least important
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

# Export for Power BI / README
feature_importance.to_csv("outputs/feature_importance.csv", index=False)

print("Feature importance file exported successfully!")
feature_importance.head(15)

Feature importance file exported successfully!


,Feature,Importance
22,Rolling_Mean_4,0.515902
17,Sales_Lag_1,0.342397
19,Sales_Lag_4,0.028550
23,Rolling_Mean_8,0.017767
18,Sales_Lag_2,0.015483
12,IsHoliday,0.009938
14,Month,0.007658
15,Week,0.006320
26,Type_B,0.005753
25,Rolling_Std_8,0.005075


In [34]:
import os

print("Files inside outputs folder:")
print(os.listdir("outputs"))

Files inside outputs folder:
['model_metrics.csv', 'inventory_planning.csv', 'feature_importance.csv', 'forecast_results.csv']


In [35]:
print("Forecast results:")
print(pd.read_csv("outputs/forecast_results.csv").shape)

print("\nModel metrics:")
print(pd.read_csv("outputs/model_metrics.csv").shape)

print("\nInventory planning:")
print(pd.read_csv("outputs/inventory_planning.csv").shape)

print("\nFeature importance:")
print(pd.read_csv("outputs/feature_importance.csv").shape)

Forecast results:
(23604, 10)

Model metrics:
(5, 2)

Inventory planning:
(3115, 19)

Feature importance:
(28, 2)


In [36]:
import shutil

shutil.make_archive("walmart_project_outputs", "zip", "outputs")

print("Zip file created: walmart_project_outputs.zip")

Zip file created: walmart_project_outputs.zip


In [37]:
from google.colab import files

files.download("walmart_project_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>